In [1]:
import jax 
import jax.numpy as jnp
import WDM

In [2]:
seed = 1234
key = jax.random.key(seed)

wdm = WDM.code.discrete_wavelet_transform.WDM.WDM_transform(
    dt=0.5,
    Nf=16,
    N=512,
    q=8,
    calc_m0=True,
)

key, subkey = jax.random.split(key)
x = jax.random.normal(subkey, shape=(wdm.N,))

w = wdm.forward_transform_exact(x)

# Keep only m=0
w_m0 = jnp.zeros_like(w)
w_m0 = w_m0.at[:, 0].set(w[:, 0])

x_exact = wdm.inverse_transform_exact(w_m0)
x_fft = wdm.inverse_transform_fft(w_m0)

max_error = jnp.max(jnp.abs(x_fft - x_exact))
relative_error = (
    jnp.linalg.norm(x_fft - x_exact)
    / jnp.linalg.norm(x_exact)
)

print(f"Maximum error:  {max_error:.3e}")
print(f"Relative error: {relative_error:.3e}")

Maximum error:  2.220e-16
Relative error: 2.421e-16


In [3]:
w = wdm.forward_transform_exact(x)

x_exact = wdm.inverse_transform_exact(w)
x_fft = wdm.inverse_transform_fft(w)

max_error = jnp.max(jnp.abs(x_fft - x_exact))
relative_error = (
    jnp.linalg.norm(x_fft - x_exact)
    / jnp.linalg.norm(x_exact)
)

print(f"Maximum error:  {max_error:.3e}")
print(f"Relative error: {relative_error:.3e}")

Maximum error:  2.895e-13
Relative error: 5.714e-14


In [4]:
w = wdm.forward_transform_fft(x)
x_roundtrip = wdm.inverse_transform_fft(w)

max_error = jnp.max(jnp.abs(x_roundtrip - x))
relative_error = (
    jnp.linalg.norm(x_roundtrip - x)
    / jnp.linalg.norm(x)
)

print(f"Round-trip maximum error:  {max_error:.3e}")
print(f"Round-trip relative error: {relative_error:.3e}")

Round-trip maximum error:  2.665e-15
Round-trip relative error: 8.084e-16


In [7]:
q_vals = [2, 4, 8, 16]

for q in q_vals:
    wdm_q = WDM.code.discrete_wavelet_transform.WDM.WDM_transform(
        dt=0.5,
        Nf=16,
        N=512,
        q=q,
        calc_m0=True,
    )

    w_q = wdm_q.forward_transform_fft(x)

    # Compile first
    wdm_q.inverse_transform(w_q).block_until_ready()
    wdm_q.inverse_transform_fft(w_q).block_until_ready()

    print(f"q = {q}")

    print("old inverse:")
    %timeit wdm_q.inverse_transform(w_q).block_until_ready()

    print("fast FFT inverse:")
    %timeit wdm_q.inverse_transform_fft(w_q).block_until_ready()

    print()

q = 2
old inverse:
200 μs ± 6.83 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
fast FFT inverse:
24.3 μs ± 2.17 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

q = 4
old inverse:
372 μs ± 28.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
fast FFT inverse:
24.3 μs ± 538 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

q = 8
old inverse:
618 μs ± 13.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
fast FFT inverse:
24.2 μs ± 117 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

q = 16
old inverse:
1.2 ms ± 22 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
fast FFT inverse:
25.2 μs ± 850 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)

